In [ ]:
from google.colab import drive

# Google Drive を /content/drive にマウント
drive.mount('/content/drive')

%cd /content/drive/MyDrive/research/proj-spectrum_denoising_1d/spectrum_denoising_1d/examples/
!pip install pytorch_lightning

In this notebook, we check that samples from the noise model resemble real noise samples.
This version (v2) uses **positional-encoding augmented** noise data.


In [ ]:
import sys

import torch
import numpy as np

sys.path.append('../')
from noise_model.PixelCNN import PixelCNN
from utils.tools import autocorrelation, plot

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Load trained noise model (PE version).

In [ ]:
noise_model_location = f"../nm_checkpoint_pe/final_params.ckpt"
noise_model = PixelCNN.load_from_checkpoint(noise_model_location).eval().to(device)

Load real noise samples (PE version).

In [ ]:
at_pbs_location = f"./../../data/for_1d_denoising/pbs_1d_pe.npy"
at_pbs = np.load(at_pbs_location)
print("at_pbs.shape", at_pbs.shape)
# Expected: (n_samples, 1, channel, 1+d_model) e.g. (100, 1, 200, 5)

# The mean of noise should be zero (noise channel only).
noise_ch_mean = np.mean(at_pbs[..., 0])
at_pbs[..., 0] = at_pbs[..., 0] - noise_ch_mean

Set sample shape and sample from noise model.

For PE-conditioned sampling, we extract the PE features from the real data
and pass them to `noise_model.sample()`.

In [ ]:
n = 1
W = at_pbs.shape[-2]   # channel dimension (e.g. 200)

# Extract PE features [n, W, d_model] from real data
pe_features = torch.tensor(
    at_pbs[:n, 0, :, 1:],   # drop the scalar noise channel (index 0)
    dtype=torch.float,
)

sample_shape = [n, W]
sample = noise_model.sample(sample_shape, pe_features=pe_features).detach().cpu().numpy()
print("sample.shape", sample.shape)   # (n, 1, W)

Visually compare real noise (scalar channel) against noise model sample.

In [ ]:
# Real noise: scalar channel only
real_noise = at_pbs[0, 0, :, 0]
generated_noise = sample[0, 0]

plot([real_noise, generated_noise],
     titles=['Real noise', 'Generated noise'],
     figsize=(20, 5))

Compare autocorrelation of real vs generated noise.

In [ ]:
autocorrelation(
    [real_noise, generated_noise],
    max_lag=50,
    titles=['Real noise', 'Generated noise'],
)